# AdaptAssist — Round 2 Training with Real Datasets
**Theme**: World Modeling — Personalized Tasks (#3.2)
**Sub-theme**: Patronus AI — Consumer Workflows with Schema Drift

### Datasets used:
1. `glaiveai/glaive-function-calling-v2` — 113k real tool-call conversations
2. `HuggingFaceH4/helpful_instructions` — Personal task instruction pairs
3. `Salesforce/xlam-function-calling-60k` — Structured API call examples
4. `AdaptAssist environment trajectories` — Expert agent demonstrations
5. `Drift-augmented samples` — Schema adaptation demonstrations (unique to us)

### Pipeline:
1. Install + setup
2. Load and process real datasets
3. Generate environment expert trajectories
4. Combine into unified SFT dataset (~7500 samples)
5. SFT warmup on Llama 3.2 3B
6. GRPO RL against live environment
7. Evaluation + plots

In [2]:
# ── Cell 1: Install ───────────────────────────────────────────────
!pip install openenv unsloth trl transformers datasets peft accelerate bitsandbytes matplotlib -q
print('Done.')

Done.


In [ ]:
# -- Cell 2: Clone or update project from GitHub ----------------
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Sourav1331/adaptassist-openenv"
BRANCH = "main"

repo_name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
repo_path = Path("/content") / repo_name

if not repo_path.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL], check=True, cwd="/content")
else:
    subprocess.run(["git", "fetch", "origin"], check=True, cwd=str(repo_path))
    subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True, cwd=str(repo_path))
    subprocess.run(["git", "clean", "-fd"], check=True, cwd=str(repo_path))

os.chdir(repo_path)
print("Working directory:", os.getcwd())

subprocess.run(["pip", "install", "-r", "requirements.txt", "-q"], check=True)
print("Requirements installed.")

os.makedirs("data", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("plots", exist_ok=True)
print("Project ready.")

In [ ]:
# ── Cell 3: Load real HuggingFace datasets ─────────────────────────
import sys
sys.path.insert(0, '.')

from data.dataset_loader import build_combined_dataset, convert_for_trl

dataset = build_combined_dataset(
    output_path='data/combined_sft.json',
    max_glaive=3000,
    max_helpful=2000,
    max_xlam=2000,
    n_drift_augmented=500,
    env_sft_path='data/sft_dataset.json',
)

print(f'\nTotal dataset: {len(dataset)} samples')
from collections import Counter
sources = Counter(s.get('source','?') for s in dataset)
for src, count in sources.most_common():
    print(f'  {src}: {count}')

In [ ]:
# ── Cell 4: Generate environment expert trajectories ───────────────
!python data/generate_sft_data.py \
    --n 60 \
    --output data/sft_dataset.json \
    --difficulties 1 2 \
    --seed 42

from data.dataset_loader import build_combined_dataset
dataset = build_combined_dataset(
    output_path='data/combined_sft.json',
    env_sft_path='data/sft_dataset.json',
)
print(f'Final dataset with env trajectories: {len(dataset)} samples')

In [ ]:
# ── Cell 5: Inspect dataset samples ───────────────────────────────
import json, random
from collections import Counter

with open('data/combined_sft.json') as f:
    all_data = json.load(f)

print(f'Dataset size: {len(all_data)} samples')
print(f'Avg messages per sample: {sum(len(s["messages"]) for s in all_data)/len(all_data):.1f}')
print(f'Avg reward: {sum(s.get("reward",1) for s in all_data)/len(all_data):.3f}')

for source in ['glaive', 'helpful_instructions', 'xlam', 'drift_augmented', 'env']:
    samples = [s for s in all_data if source in s.get('source','')]
    if samples:
        s = random.choice(samples)
        print(f'\n--- {source} sample ---')
        print(f'Messages: {len(s["messages"])} | Reward: {s.get("reward",1)}')
        if len(s['messages']) > 1:
            print(f'User: {s["messages"][1]["content"][:120]}...')
        if len(s['messages']) > 2:
            print(f'Assistant: {s["messages"][2]["content"][:120]}...')

In [ ]:
# ── Cell 6: Load model with Unsloth ───────────────────────────────
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-3B-Instruct',
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
# ── Cell 7: SFT Warmup ─────────────────────────────────────────────
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
import json

with open('data/combined_sft.json') as f:
    raw = json.load(f)

hf_dataset = Dataset.from_list([{'messages': s['messages']} for s in raw])
split = hf_dataset.train_test_split(test_size=0.05, seed=42)
print(f'Train: {len(split["train"])} | Eval: {len(split["test"])}')

sft_config = SFTConfig(
    output_dir='checkpoints/sft',
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=100,
    save_steps=200,
    max_seq_length=4096,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    args=sft_config,
)

print('Starting SFT training (~45-60 min on A10G)...')
trainer.train()

model.save_pretrained('checkpoints/sft')
tokenizer.save_pretrained('checkpoints/sft')
print('SFT complete. Saved to checkpoints/sft')

In [ ]:
# ── Cell 8: GRPO RL Training ──────────────────────────────────────
import requests, time, json
from trl import GRPOConfig, GRPOTrainer

ENV_URL = 'https://huggingface.co/spaces/Souravdanyal/adaptassist-env'  # ← verify this is your env Space URL

def format_reward(completions, **kwargs):
    rewards = []
    for c in completions:
        text = c[0]['content'] if isinstance(c, list) else c
        try:
            t = text.strip()
            if t.startswith('```'): t = t.split('\n',1)[1].rsplit('```',1)[0]
            d = json.loads(t)
            rewards.append(1.0 if 'tool' in d and 'params' in d else 0.0)
        except:
            rewards.append(0.0)
    return rewards

def drift_detection_reward(completions, **kwargs):
    rewards = []
    for c in completions:
        text = c[0]['content'] if isinstance(c, list) else c
        try:
            d = json.loads(text.strip().lstrip('```json').rstrip('```'))
            rewards.append(0.3 if d.get('tool') == 'detect_schema_change' else 0.0)
        except:
            rewards.append(0.0)
    return rewards

def env_reward(completions, prompts, **kwargs):
    rewards = []
    for completion, prompt in zip(completions, prompts):
        text = completion[0]['content'] if isinstance(completion, list) else completion
        try:
            t = text.strip()
            if t.startswith('```'): t = t.split('\n',1)[1].rsplit('```',1)[0]
            action = json.loads(t)

            difficulty = 'easy'
            for d in ['easy','medium','hard']:
                if d in str(prompt).lower():
                    difficulty = d
                    break

            obs = requests.post(f'{ENV_URL}/reset',
                json={'difficulty': difficulty}, timeout=15).json()['observation']

            result = requests.post(f'{ENV_URL}/step',
                json={'tool': action.get('tool','finish'),
                      'params': action.get('params',{})}, timeout=15).json()

            rewards.append(result.get('reward', 0.0))
        except:
            rewards.append(0.0)
        time.sleep(0.05)
    return rewards

# Wake up environment Space first
print('Waking up environment Space...')
try:
    r = requests.get(ENV_URL, timeout=30)
    print(f'Environment ready: {r.status_code}')
except:
    print('WARNING: Could not reach environment. Check your ENV_URL.')

from datasets import Dataset
grpo_samples = [{'prompt': s['messages'][:2]} for s in raw if len(s['messages']) >= 2]
grpo_dataset = Dataset.from_list(grpo_samples).train_test_split(test_size=0.1)

grpo_config = GRPOConfig(
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=1024,
    max_completion_length=512,
    max_steps=300,
    save_steps=100,
    output_dir='checkpoints/grpo',
    report_to='none',
)

grpo_trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward, env_reward, drift_detection_reward],
    args=grpo_config,
    train_dataset=grpo_dataset['train'],
)

print('Starting GRPO RL training (300 steps, ~2hr on A10G)...')
grpo_trainer.train()
print('GRPO training complete.')

In [ ]:
# ── Cell 9: Reward curves ─────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

log = grpo_trainer.state.log_history
steps   = [l['step'] for l in log if 'rewards/env_reward' in l]
env_r   = [l['rewards/env_reward'] for l in log if 'rewards/env_reward' in l]
fmt_r   = [l.get('rewards/format_reward', 0) for l in log if 'rewards/env_reward' in l]
drift_r = [l.get('rewards/drift_detection_reward', 0) for l in log if 'rewards/env_reward' in l]

def roll(lst, w=15):
    return [sum(lst[max(0,i-w):i+1])/len(lst[max(0,i-w):i+1]) for i in range(len(lst))]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('AdaptAssist — GRPO Training on Real Datasets', fontsize=13, fontweight='bold')

for ax, vals, label, color in [
    (axes[0], env_r,   'Environment Reward (test pass rate)', '#1D9E75'),
    (axes[1], fmt_r,   'Format Reward (valid JSON output)',   '#378ADD'),
    (axes[2], drift_r, 'Drift Detection Reward',              '#D85A30'),
]:
    ax.scatter(steps, vals, alpha=0.3, s=12, color=color)
    if len(vals) > 5:
        ax.plot(steps, roll(vals), color=color, linewidth=2.5)
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Reward')
    ax.set_title(label)
    ax.set_ylim(-0.05, 1.1)

plt.tight_layout()
plt.savefig('plots/reward_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved plots/reward_curves.png')

In [ ]:
# ── Cell 10: Results summary ──────────────────────────────────────
from collections import Counter

print('=== DATASET SUMMARY ===')
print(f'Total SFT samples: {len(all_data)}')
for src, count in Counter(s.get("source","?") for s in all_data).most_common():
    print(f'  {src}: {count} ({count/len(all_data)*100:.1f}%)')

print('\n=== TRAINING RESULTS ===')
if env_r:
    print(f'Initial env reward:  {env_r[0]:.3f}')
    print(f'Final env reward:    {env_r[-1]:.3f}')
    print(f'Peak env reward:     {max(env_r):.3f}')
    print(f'Improvement:         +{env_r[-1]-env_r[0]:.3f}')
else:
    print('No reward logs found — check grpo_trainer.state.log_history')

In [ ]:
# ── Cell 11: Push to Hub ──────────────────────────────────────────
import os
from huggingface_hub import login, HfApi

HF_TOKEN = ''  # ← paste your HF token here

login(token=HF_TOKEN)

# Push trained model
model.save_pretrained('adaptassist-llama3-grpo')
tokenizer.save_pretrained('adaptassist-llama3-grpo')
model.push_to_hub('Sourav1331/adaptassist-llama3-grpo')
tokenizer.push_to_hub('Sourav1331/adaptassist-llama3-grpo')
print('Model pushed.')

# Push reward curve plot back to your GitHub repo env Space
api = HfApi()
api.upload_file(
    path_or_fileobj="plots/reward_curves.png",
    path_in_repo="plots/reward_curves.png",
    repo_id="Sourav1331/adaptassist-openenv",
    repo_type="space",
    token=HF_TOKEN
)
print('Plot pushed to Space repo.')
print('All done!')